<div style="background: #86d1f1ff; border-radius: 5px; padding: 1rem; margin-bottom: 1rem">
<img src="https://store.utec.edu.pe/files/Recursos/logo-utec-h.png" alt="Banner" width="150" />   
<div style="font-weight: bold; color: #434549ff; float: right "><u style="font-size: 28px;">Base de Datos II</u> <br />
<span style="float:right"> Profesor Heider Sanchez</span> <br /> 
<span style="float:right">  2026 - 1 </span>   
</div> </div>

# Laboratorio 8.2: Similitud de Coseno e Indice Invertido

> **Prof. Heider Sanchez**  

## Introducción

Este laboratorio extiende las capacidades desarrolladas en el laboratorio 7.1, enfocándose en técnicas avanzadas de recuperación de información. Utilizaremos los Bag of Words previamente generados para implementar dos funcionalidades esenciales en los motores de búsqueda modernos: el **Índice Invertido** para recuperación eficiente de documentos, y la **Similitud de Coseno** para resultados ordenados por relevancia.


### Objetivos
- Implementar la conexión a PostgreSQL para extraer id, contenido y bag_of_words de los documentos.
- Construir el índice invertido (posting lists), calcular estadísticas IDF y la norma vectorial de cada documento.
- Implementar búsquedas booleanas (AND, OR y AND-NOT) con complejidad O(n+m) usando el índice invertido.
- Implementar búsquedas rankeadas usando la Similitud de Coseno:
  - Procesar consultas en lenguaje natural aplicando tokenización y cálculo del TF.
  - Utilizar el índice invertido para obtener los documentos que intersectan con la query.
  - Calcular similitud de coseno con TF-IDF entre la query y los documentos recuperados.
  - Devolver los top-k resultados ordenados por relevancia.
  - **Evitar usar la representación vectorial dispersa.**

In [14]:
import psycopg2
import pandas as pd
import json

def connect_db():
    conn = psycopg2.connect(
        dbname="lab8db",
        user="postgres",
        password="123456",
        host="localhost",
        port="5000"
    )
    return conn

def fetch_data():
    conn = connect_db()
    query = "SELECT id, contenido, bag_of_words FROM noticias;"
    df = pd.read_sql(query, conn)
    # acá solo comente esa línea porque ya tenemos el bag_of_words como un diccionario en la base de datos pq en el anteiror lab lo guardaron así en la parte 2 y 3 xd
    #df['bag_of_words'] = df['bag_of_words'].apply(json.loads) 
    conn.close()
    return df

noticias_df = fetch_data()

C:\Users\cosea\AppData\Local\Temp\ipykernel_20532\3484805427.py:18: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


In [15]:
noticias_df

,id,contenido,bag_of_words
0,55,Pasar la tarjeta (o últimamente el teléfono mó...,"{'my': 1, 'qr': 2, 'ali': 1, 'app': 3, 'aqu': ..."
1,705,Más de 600 jóvenes de nueve nacionalidades dif...,"{'of': 1, 'are': 1, 'aws': 3, 'bas': 1, 'cas':..."
2,65,La Medicina se encuentra en un punto de transf...,"{'cb': 1, 'ia': 3, 'of': 1, 'are': 1, 'aug': 1..."
3,79,En el día de cierre del Claro Tech Summit Micr...,"{'z': 1, 'cre': 1, 'dat': 1, 'dia': 1, 'gam': ..."
4,86,"Para impulsar la descarbonización del planeta,...","{'agu': 1, 'baj': 1, 'bas': 2, 'car': 1, 'cas'..."
...,...,...,...
1212,1213,En la vida de toda empresa emergente llega un ...,"{'x': 3, 'bas': 1, 'cas': 2, 'cre': 2, 'dam': ..."
1213,1214,La espiral alcista de los precios continúa y g...,"{'dud': 1, 'dur': 1, 'jef': 1, 'med': 1, 'mes'..."
1214,1215,Las grandes derrotas nacionales son experienci...,"{'xi': 1, 'cas': 1, 'dej': 1, 'gas': 1, 'her':..."
1215,1216,BBVA ha alcanzado un acuerdo de colaboración c...,"{'bhh': 9, 'cab': 1, 'cuy': 1, 'dud': 1, 'hub'..."


## 1. (6 puntos) Construcción del Indice Invertido 

A partir de los  `bag of words` almacenados en la base de datos  (Laboratorio 8.1), se debe construir un índice invertido y conservarlo en un diccionario de Python para su eficiente recuperación.

In [16]:
noticias_df = fetch_data()

C:\Users\cosea\AppData\Local\Temp\ipykernel_20532\3484805427.py:18: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


In [21]:
import math
import re
from collections import Counter

class InvertedIndex:
    def __init__(self):
        self.index = {}
        self.idf = {}
        self.length = {}

    def build_from_db(self):
        # Leer desde PostgreSQL todos los bag of words
        # Construir el índice invertido, el idf y la norma (longitud) de cada documento
        
        """
        indice  = {
            "word1": [("doc1", tf1), ("doc2", tf2), ("doc3", tf3)],
            "word2": [("doc2", tf2), ("doc4", tf4)],
            "word3": [("doc3", tf3), ("doc5", tf5)],
        } 
        idf  = {
            "word1": 3,
            "word2": 2,
            "word3": 2,
        } 
        length = {
            "doc1": 15.5236,
            "doc2": 10.5236,
            "doc3": 5.5236,
        }
        """
        df = fetch_data()
        self.N = len(df)
        df_counts = {}
        
        # Contamos en cuántos documentos aparece cada palabra (df)
        for _, row in df.iterrows():
            bow = row['bag_of_words']
            for word in bow.keys():
                df_counts[word] = df_counts.get(word, 0) + 1
                
        # Calculamos idf para cada término (log10(N / df))
        for word, count in df_counts.items():
            self.idf[word] = math.log10(self.N / count)
            
        # Construimos el índice invertido y las normas de los documentos
        for _, row in df.iterrows():
            doc_id = row['id']
            bow = row['bag_of_words']
            
            doc_length_sq = 0.0
            
            for word, tf in bow.items():
                if word not in self.index:
                    self.index[word] = []
                self.index[word].append((doc_id, tf))
                # Calcular peso TF-IDF para la norma del documento
                tf_weight = 1 + math.log10(tf) if tf > 0 else 0
                w_td = tf_weight * self.idf[word]
                
                doc_length_sq += w_td ** 2
                
            self.length[doc_id] = math.sqrt(doc_length_sq)
        pass
    
    def L(self, word):
        # Retorna la lista de documentos que contienen a word 
        return self.index.get(word, [])

    def cosine_search(self, query, top_k=5):  
        score = {}
        # No es necesario usar vectores numericos del tamaño del vocabulario
        # Guiarse del algoritmo visto en clase
        # Se debe calcular el tf-idf de la query y de cada documento
        # La query se debe procesar  al igual como se procesaron los documentos (bag of words)
        
        query = re.sub(r'[^\w\s]', '', query.lower())
        query_bow = Counter(query.split())
        query_length_sq = 0.0
        for word, q_tf in query_bow.items():
            if word in self.idf:
                q_tf_weight = 1 + math.log10(q_tf) if q_tf > 0 else 0
                w_tq = q_tf_weight * self.idf[word]
                
                query_length_sq += w_tq ** 2
                
                posting_list = self.L(word) # lista de documentos que contienen a word 
                
                for doc_id, doc_tf in posting_list:
                    doc_tf_weight = 1 + math.log10(doc_tf) if doc_tf > 0 else 0
                    w_td = doc_tf_weight * self.idf[word]
                    
                    # Acumulamos el producto punto
                    score[doc_id] = score.get(doc_id, 0.0) + (w_tq * w_td)
                    
        query_length = math.sqrt(query_length_sq)
        
        # Normalización Coseno: dividir por las normas
        if query_length > 0:
            for doc_id in score:
                score[doc_id] = score[doc_id] / (query_length * self.length[doc_id])
        
        # Ordenar el score resultante de forma descendente
        result = sorted(score.items(), key= lambda tup: tup[1], reverse=True)
        # retornamos los k documentos mas relevantes (de mayor similitud a la query)
        return result[:top_k]
    
    #this implementation is going to be used for the question 3:
    def showDocument(self, doc_id):
        # Retorna el contenido del documento dado su id
        conn = connect_db()
        query = "SELECT contenido FROM noticias WHERE id = %s;"
        cursor = conn.cursor()
        cursor.execute(query, (doc_id,))
        result = cursor.fetchone()
        conn.close()
        return result[0] if result else None 

In [22]:
#Esto es mi prueba si quieren lo pueden borrar despues
buscador = InvertedIndex()
buscador.build_from_db()
resultados = buscador.cosine_search("dol ia cas", top_k=5)
print(resultados)

C:\Users\cosea\AppData\Local\Temp\ipykernel_20532\3484805427.py:18: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


[(924, 0.19083116465788505), (173, 0.12499104867009081), (1018, 0.12206731558574987), (314, 0.10830455719464036), (486, 0.10830455719464036)]


## 2. (6 puntos) Consultas Booleanas usando el indice invertido

Implementar búsquedas booleanas utilizando el índice invertido construido anteriormente. La búsqueda debe:

- Soportar los operadores básicos:
    - AND: intersección de documentos
    - OR: unión de documentos
    - AND-NOT: diferencia de documentos
- Procesar consultas como:
    - "sostenibilidad AND ambiente AND renovable"
    - "tecnología AND (banca OR finanzas)"
    - "economía AND-NOT inflación"    

####  Pruebas funcionales

In [ ]:
idx = InvertedIndex()
idx.build_from_db()

def AND(list1, list2):
    # Implementar la intersección de dos listas O(n +m)
    pass

def OR(list1, list2):
    # Implementar la unión de dos listas O(n +m)
    pass

def AND_NOT(list1, list2):
    # Implementar la diferencia de dos listas O(n +m)
    pass

# Prueba 1
result = AND(idx.L("sostenibilidad"), AND(idx.L("ambiente"), idx.L("renovables")))
print("sostenibilidad AND ambiente AND renovable: ", idx.showDocuments(result))

# Prueba 2
result = AND(idx.L("tecnología"), OR(idx.L("banca"), idx.L("finanzas")))
print("tecnología AND (banca OR finanzas): ", idx.showDocuments(result))

# Prueba 3
result = AND_NOT(idx.L("economía"), idx.L("inflación"))
print("economía AND-NOT inflación: " , idx.showDocuments(result))

# Agregar dos pruebas mas combinando los operadores AND, OR, AND_NOT

## 3. (8 puntos) Similitud de Coseno usando el indice invertido
Implementar búsqueda por similitud de coseno aprovechando el índice invertido:

- Proceso de búsqueda:
    - Recibe una consulta en lenguaje natural y un parámetro top_k
    - Utiliza el índice invertido para identificar documentos candidatos
    - Calcula similitud de coseno solo con los documentos relevantes utilizando los pesos TF-IDF
    - Retorna los top-k documentos más similares

<img src="https://1drv.ms/i/c/0c2923df9f1f816f/IQSELMi5qcbqS7lsy5sn8ZLpAZ3G2ciXdabecVJ0vhKoL78" width="500" align="" />

####  Pruebas funcionales

In [25]:
test_queries = [
    "¿Cuáles son las últimas innovaciones en la banca digital y la tecnología financiera?",
    "evolución de la inflación y el crecimiento de la economía en los últimos años",
    "avances sobre sostenibilidad y energías renovables para el medio ambiente"
]
buscador = InvertedIndex()
buscador.build_from_db()

for test in test_queries:    
    results = buscador.cosine_search(test, top_k=5)
    print(f"Top 5 documentos más similares:")    
    for doc_id, score in results:
        print(f"Doc {doc_id}: {score:.3f}: ", buscador.showDocument(doc_id))

C:\Users\cosea\AppData\Local\Temp\ipykernel_20532\3484805427.py:18: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


Top 5 documentos más similares:
Doc 167: 0.108:  La jornada tuvo lugar el viernes 23 de septiembre en el marco de la Maratón Nacional de Lectura y contó con la participación de jóvenes, docentes, familias e instituciones educativas de todo Argentina.
BBVA en Argentina acompañó a niños y niñas lectores de todo el país en la 20° edición de la Maratón Nacional de Lectura de Fundación Leer. El evento, que año a año busca celebrar la lectura, se realizó de forma virtual y tuvo como lema “Leer te empodera: palabras, futuro e imaginación”.
“Nos parece sumamente importante impulsar esta iniciativa para promover el hábito de la lectura, como un factor clave en la educación de los niños y niñas del país. En BBVA entendemos que la educación es primordial y, en este caso, hacerlo trabajando con la lectura como herramienta de integración social, nos permite crear nuevas y mejores oportunidades para los jóvenes”, expresó María Martha Deleonardis, subgerente de Negocio Responsable de BBVA en Argentin